In [ ]:
!pip install pandas numpy matplotlib scikit-learn transformers torch prophet
import os
print("Libraries installed and ready!")

# ***Cell 1 Imports & Setup***

In [ ]:
import os
import json
import shutil
import torch
import torch.nn as nn
from IPython.display import display, Markdown
from google.colab import files

export_dir = "/content/Frozen_Pipeline_Day22"
os.makedirs(export_dir, exist_ok=True)

# ***Cell 2 Define Final GRU Model & Save Weights***

In [ ]:
class FinalPrognosticGRU(nn.Module):
    def __init__(self, input_dim=153, hidden_dim=64, num_layers=2, output_dim=1, dropout=0.1):
        super(FinalPrognosticGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.fc1(out[:, -1, :])
        out = self.relu(self.dropout(out))
        return self.fc2(out).squeeze()

frozen_model = FinalPrognosticGRU()
torch.save(frozen_model.state_dict(), os.path.join(export_dir, "frozen_model_weights.pth"))

# ***Cell 3 Save Frozen Pipeline Config***

In [ ]:
pipeline_config = {
    "preprocessing": {
        "sequence_window": 10,
        "feature_extraction_limit": "first_500_seconds_only",
        "signals_used": ["voltage", "current", "temperature"],
        "scaler": "StandardScaler",
        "scaler_fit_scope": "training_cells_only"
    },
    "data_split": {
        "methodology": "GroupKFold",
        "group_key": "battery_id",
        "objective": "prevent_identity_leakage"
    },
    "evaluation": {
        "primary_metrics": ["MAE", "RMSE", "MAPE", "R2"],
        "target_variable": "State_of_Health_SOH"
    }
}

with open(os.path.join(export_dir, "frozen_methodology.json"), "w") as f:
    json.dump(pipeline_config, f, indent=4)

# ***Cell 4 Day 22 Final Model & Pipeline Rationale***

In [ ]:
rationale_text = """# Day 22: Final Model & Pipeline Selection Rationale

## 1. Selection by Validation Evidence
GRU chosen as final architecture --- across grid search, it beat Ridge and Random Forest, hitting R2 > 0.85 and cutting MAE ~50% on validation.

## 2. Robustness Assessment
Strict cross-cell isolation held; smooths regeneration spikes but tracks the non-linear degradation trend better than LSTM, with less overfitting to training cells.

## 3. Appropriate Simplicity
~25% fewer params than LSTM (simpler gating, no separate cell state) --- acts as a natural regularizer for the small NASA dataset; Transformers judged too heavy for 10-step windows.

## 4. Frozen Pipeline
- **Preprocessing:** first 500s of discharge only; full-cycle features banned (leakage prevention)
- **Split Strategy:** GroupKFold on `battery_id`, frozen
- **Artefacts:** pipeline config + initial model weights saved and zipped for reproducibility
"""

with open(os.path.join(export_dir, "Technical_Rationale.md"), "w") as f:
    f.write(rationale_text)

display(Markdown(rationale_text))

# ***Cell 5 Package & Download Artefacts***

In [ ]:
shutil.make_archive(export_dir, 'zip', export_dir)
files.download(f"{export_dir}.zip")